In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [2]:
import re, time, unicodedata, json, requests
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# ── Pilih backend ────────────────────────────────────────
BACKEND = 'groq'  

# Groq (cloud)
GROQ_API_KEY = GROQ_API_KEY.strip()
GROQ_MODEL   = GROQ_MODEL = "llama-3.3-70b-versatile" 

# ── Context window ───────────────────────────────────────
MAX_CHARS = 3500  # ~75% dari 4096 token Llama 3 8B
                  # Naikkan ke 6000+ untuk Llama 3.1 70B/405B

# Data Paths
DATA_PATH = Path('../../datasets')
PROCESSED_PATH = DATA_PATH / 'processed_data/processed.pkl'
VOCABS_PATH = DATA_PATH / 'processed_data/vocabs.pkl'
SPLITS_PATH = DATA_PATH  / 'split_data/split_indices.json'

# ── Entity labels ────────────────────────────────────────
ENTITY_LABELS = [
    'Name', 'Email Address', 'Years of Experience',
    'Companies worked at', 'Designation', 'Skills',
    'Location', 'College Name', 'Degree', 'Graduation Year', 'UNKNOWN'
]

print('Config loaded OK')
print(f'Backend : {BACKEND} | Labels: {len(ENTITY_LABELS)}')

Config loaded OK
Backend : groq | Labels: 11


In [3]:
import pickle

with open(PROCESSED_PATH, 'rb') as f: 
    processed = pickle.load(f)
with open(VOCABS_PATH, 'rb') as f: 
    vocabs = pickle.load(f)
with open(SPLITS_PATH, 'rb') as f: 
    splits = json.load(f)

idx_train = splits['idx_train']
idx_val= splits['idx_val']
idx_test = splits['idx_test']


ENTITY_LABELS = vocabs['Entity_labels'] + ['UNKNOWN']
id2labels = vocabs['id2label']

print(f'Total records: {len(idx_train)} / {len(idx_val)} / {len(idx_test)}')
print(f'Entity labels: {ENTITY_LABELS}')
p0 = processed[idx_test[0]]
print(f'TEst record {idx_test[0]}:\n{p0["content"][:200]}')
print('5 first span')
for s, e, lbl in p0['spans'][:5]:
    print(f'[{s}:{e}] {lbl:<25} {repr(p0["content"][s:e])}')
    

Total records: 154 / 33 / 33
Entity labels: ['Name', 'Designation', 'Companies worked at', 'Location', 'Email Address', 'College Name', 'Degree', 'Graduation Year', 'Skills', 'Years of Experience', 'UNKNOWN']
TEst record 137:
Dushyant Bhatt
BI / Big Data/ Azure

Hyderabad-Deccan, Telangana, Telangana - Email me on Indeed: indeed.com/r/Dushyant-
Bhatt/140749dace5dc26f

• 10+ years of Experience in Designing, Development, Ad
5 first span
[0:14] Name                      'Dushyant Bhatt'
[37:46] Location                  'Hyderabad'
[98:143] Email Address             'indeed.com/r/Dushyant-\nBhatt/140749dace5dc26f'
[729:738] Companies worked at       'Microsoft'
[1753:1771] Designation               'Software Engineer\n'


In [4]:
from collections import defaultdict

def extract_ground_truth(record: dict) -> dict:
    gt = defaultdict(list)
    content = record['content']

    for start, end, label in record['spans']:
        text = content[start:end].strip().lower()

        if text and text not in gt[label]:
            gt[label].append(text)

    return dict(gt)

In [5]:
def clean_resume_text(text : str) -> str: 
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(f'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(
        r'[\u2022\u00b7\u25aa\u25cf\u25a0\u25e6\u25cb\u25b8\u25ba]', 
        '-', 
        text)
    text= re.sub(r'[^\S\n]+', ' ', text)
    return text.strip()

def truncate_if_needed(text : str, max_chars : int= MAX_CHARS) -> list: 
    if len(text) <= max_chars: 
        return [text]
    chunks, current = [], ''
    for para in text.split('\n\n'): 
        if len(current) + len(para) + 2 <= max_chars: 
            current += para + '\n\n'
        else: 
            if current: chunks.append(current.strip())
            current = para + '\n\n'

    if current.strip() : chunks.append(current.strip())
    return chunks if chunks else [text[:max_chars]]
lengths = [len(processed[i]['content']) for i in idx_test] 
print(f'Test set resume length: min: {min(lengths)} max: {max(lengths)} mean: {sum(lengths) / len(lengths)}')
print(f'Resume > MAX_CHARS ({MAX_CHARS}: {sum(1 for l in lengths if l > MAX_CHARS)})')
    

Test set resume length: min: 452 max: 9018 mean: 3989.0
Resume > MAX_CHARS (3500: 17)


In [6]:

system_prompt = """You are an expert Named Entity Recognition (NER) system specialized in extracting structured information from resumes and professional profiles.

You MUST follow these rules strictly:
1. Extract ONLY entities explicitly present in the text. Do NOT infer or hallucinate information.
2. Assign exactly ONE label per entity from the allowed label set.
3. If an entity does not clearly belong to any defined label, assign the label UNKNOWN.
4. Return your response as a valid JSON object. No extra explanation, no markdown, no preamble.
5. Each key in the JSON must map to a list of unique string values.
6. Normalize values: strip extra whitespace, proper nouns capitalized.

ALLOWED LABELS:
- Company worked at → Name of a company or organization the person worked at
- Designation → Job title or role
- Skills → Technical skill, tool, language, framework, or soft skill
- Location → City, state, country, or geographic region
- College Name → Name of a university, college, or educational institution
- Degree → Academic degree or qualification
- Graduation Year → Year of graduation (4-digit year)
- Email Address → Valid email address
- Name → Full name of the candidate
- Years of Experience → Duration of professional experience
- UNKNOWN → Entity present but does not fit any label above

OUTPUT FORMAT (strict JSON):
{
  "Name": [], "Email Address": [], "Years of Experience": [], "Company worked at": [],
  "Designation": [], "Skills": [], "Location": [], "College Name": [],
  "Degree": [], "Graduation Year": [], "UNKNOWN": []
}"""
def build_prompt(resume_text: str) -> str:
    return (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}\n\n"
        f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        f"Extract all named entities from the following resume text. Return only the JSON object.\n\n"
        f'RESUME TEXT:\n"""\n{resume_text}\n"""\n\n'
        f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )


In [7]:
def parse_output(raw_text : str) -> dict:
    raw_text = raw_text.strip()
    try: 
        result = json.loads(raw_text)
    except json.JSONDecodeError: 
        match = re.search('\{[\s\S]*\}', raw_text)
        try: 
            result = json.loads(match.group()) if match else {}
        except: 
            result = {}
    for label in ENTITY_LABELS: 
        result.setdefault(label, []) 
    for key in list(result.keys()): 
        if not isinstance(result[key], list): 
            result[key] = [str(result[key])] if result[key] else []
        result[key] = [str(v).strip() for v in result[key] if str(v).strip()]
    return result

def merge_chunks(results: list) -> dict: 
    merged = defaultdict(set)
    for r in results: 
        for label, vals in r.items(): 
            for v in vals: merged[label].add(v.lower().strip())
    return {l:sorted(list(v)) for l, v in merged.items()}
    


In [8]:
def call_groq(resume_text: str, retries=2) -> str:
    import groq
    import time

    client = groq.Groq(api_key=GROQ_API_KEY)

    msgs = [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': (
                'Extract entities. Return ONLY JSON.\n\n'
                'RESUME:\n"""\n' + resume_text + '\n"""'
            )
        }
    ]

    for i in range(retries + 1):
        try:
            r = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=msgs,
                temperature=0.0,
                max_tokens=1024
            )
            return r.choices[0].message.content

        except Exception as e:
            if i == retries:
                raise

            time.sleep(2 ** i)


In [9]:
def extract_entities(resume_text: str) -> dict:
    cleaned = clean_resume_text(resume_text)
    chunks = truncate_if_needed(cleaned)

    results = []
    for chunk in chunks: 
        raw = call_groq(chunk)
        results.append(parse_output(raw))

    return merge_chunks(results) if len(chunks) > 1 else results[0]

In [10]:
record = processed[idx_test[0]]
gt = extract_ground_truth(record)

print(f'Resume index: {idx_test[0]} | Length: {len(record["content"])} chars')
print('Call Llama...')

pred = extract_entities(record['content'])

print('\n=== Prediction ===')
print(json.dumps(pred, indent=2))

print('\n=== GROUND TRUTH ===')
print(json.dumps(gt, indent=2))

Resume index: 137 | Length: 6547 chars
Call Llama...

=== Prediction ===
{
  "Name": [
    "dushyant bhatt"
  ],
  "Years of Experience": [
    "10+ years"
  ],
  "Company worked at": [
    "microsoft"
  ],
  "Designation": [
    "software engineer"
  ],
  "Skills": [
    ".net api",
    "analytical",
    "angular js",
    "azure",
    "azure document db",
    "azure web job",
    "bi",
    "big data",
    "c#",
    "communication",
    "cosmos",
    "interpersonal",
    "knowledge transfer",
    "power bi",
    "problem solving",
    "project lifecycle",
    "project manager",
    "rbac",
    "sql",
    "sql server 2014",
    "ssis",
    "ssrs",
    "stored procedures",
    "tableau",
    "technical assistance",
    "u-sql",
    "web app",
    "x-flow"
  ],
  "Location": [
    "australia",
    "canada",
    "gujarat",
    "hyderabad-deccan",
    "telangana",
    "us"
  ],
  "UNKNOWN": [
    "bing",
    "biztalk",
    "icoe",
    "indeed",
    "microsoft edge",
    "microsoft rewards",

In [11]:
def normalize_val(v: str) -> str:
    return re.sub(r'\s+', ' ', v.lower().strip())


def compute_metrics(pred: dict, gt: dict) -> dict:
    results = {}

    for label in ENTITY_LABELS:
        p = set(normalize_val(v) for v in pred.get(label, []))
        g = set(normalize_val(v) for v in gt.get(label, []))

        results[label] = {
            'tp': len(p & g),
            'fp': len(p - g),
            'fn': len(g - p)
        }

    return results


def aggregate_metrics(all_results: list) -> pd.DataFrame:
    totals = {
        l: {'tp': 0, 'fp': 0, 'fn': 0}
        for l in ENTITY_LABELS
    }

    for rec in all_results:
        for l in ENTITY_LABELS:
            for k in ('tp', 'fp', 'fn'):
                totals[l][k] += rec.get(l, {}).get(k, 0)

    rows = []

    for l in ENTITY_LABELS:
        tp = totals[l]['tp']
        fp = totals[l]['fp']
        fn = totals[l]['fn']

        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

        rows.append({
            'Label': l,
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'Precision': round(p, 3),
            'Recall': round(r, 3),
            'F1': round(f1, 3)
        })

    df = pd.DataFrame(rows)

    m = df[['Precision', 'Recall', 'F1']].mean()

    df = pd.concat([
        df,
        pd.DataFrame([{
            'Label': 'MACRO AVG',
            'TP': '',
            'FP': '',
            'FN': '',
            'Precision': round(m['Precision'], 3),
            'Recall': round(m['Recall'], 3),
            'F1': round(m['F1'], 3)
        }])
    ], ignore_index=True)

    return df


print('Evaluation functions loaded OK')

Evaluation functions loaded OK


In [12]:
all_metrics = []
predictions = []
errors = []

for resume_idx in tqdm(idx_test, desc='Test set'):
    record = processed[resume_idx]
    gt = extract_ground_truth(record)

    try:
        pred = extract_entities(record['content'])
        metrics = compute_metrics(pred, gt)

        all_metrics.append(metrics)

        predictions.append({
            'resume_idx': resume_idx,
            'pred': pred,
            'gt': gt
        })

    except Exception as e:
        print(f'Error idx {resume_idx}: {e}')

        errors.append(resume_idx)
        all_metrics.append({})

print(f'\nDone. Test: {len(idx_test)} | Errors: {len(errors)}')



Test set: 100%|██████████| 33/33 [04:52<00:00,  8.86s/it]


Done. Test: 33 | Errors: 0


In [13]:
df = aggregate_metrics(all_metrics)

print(f'=== Zero-Shot Llama NER — Test Set ({len(idx_test)} samples) ===')
print(df.to_string(index=False))

try:
    display(
        df.style
        .highlight_max(subset=['F1'], color='lightgreen')
        .highlight_min(subset=['F1'], color="#c35757")
        .set_caption(
            f'Zero-Shot Llama — Test Set ({len(idx_test)} samples)'
        )
    )
except:
    pass


=== Zero-Shot Llama NER — Test Set (33 samples) ===
              Label TP  FP FN  Precision  Recall    F1
               Name 30   6  2      0.833   0.938 0.882
        Designation 22  71  9      0.237   0.710 0.355
Companies worked at  0   0 26      0.000   0.000 0.000
           Location 25 156  4      0.138   0.862 0.238
      Email Address  0   1 24      0.000   0.000 0.000
       College Name 27  43  5      0.386   0.844 0.529
             Degree 24  30  7      0.444   0.774 0.565
    Graduation Year 16  30  1      0.348   0.941 0.508
             Skills  8 694 26      0.011   0.235 0.022
Years of Experience  1  17  5      0.056   0.167 0.083
            UNKNOWN  0 195  0      0.000   0.000 0.000
          MACRO AVG                0.223   0.497 0.289


,Label,TP,FP,FN,Precision,Recall,F1
0,Name,30,6,2,0.833000,0.938000,0.882000
1,Designation,22,71,9,0.237000,0.710000,0.355000
2,Companies worked at,0,0,26,0.000000,0.000000,0.000000
3,Location,25,156,4,0.138000,0.862000,0.238000
4,Email Address,0,1,24,0.000000,0.000000,0.000000
5,College Name,27,43,5,0.386000,0.844000,0.529000
6,Degree,24,30,7,0.444000,0.774000,0.565000
7,Graduation Year,16,30,1,0.348000,0.941000,0.508000
8,Skills,8,694,26,0.011000,0.235000,0.022000
9,Years of Experience,1,17,5,0.056000,0.167000,0.083000
